# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth reviewing if it used to get real search traffic and is currently ranking poorly (position > 10) with confirmed position data. Originally included staleness (content_updated_date), but that signal was tested and found unreliable (section 1 verdict below) — dropped from the final rule.

In [19]:
import duckdb, getpass
import pandas as pd, numpy as np

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# Build prior (Feb) vs current (March) windows to define a REAL decline label
momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar,
               SUM(gsc_clicks) AS clicks_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.client_hash_id, mar.content_hash_id, mar.imp_last, mar.avg_position_mar, mar.clicks_mar,
           feb.imp_prev
    FROM mar
    LEFT JOIN feb ON mar.client_hash_id = feb.client_hash_id
                  AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL
      AND feb.imp_prev >= 100
"""
signal_df = conn.execute(momentum_query).df()
print(signal_df.shape)
signal_df.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(76837, 6)


,client_hash_id,content_hash_id,imp_last,avg_position_mar,clicks_mar,imp_prev
0,client_e547b89c05043229,content_7995404695ee1ffd,768.0,34.573578,1.0,1012.0
1,client_e547b89c05043229,content_ccbb253f142217c3,3071.0,24.332310,6.0,1598.0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,547.0,5.350029,0.0,861.0
3,client_e547b89c05043229,content_acf700633f016e5a,209.0,8.566900,0.0,245.0
4,client_e547b89c05043229,content_712e44562fed8ff2,500.0,6.870846,1.0,306.0


In [20]:
content_query = f"""
    SELECT content_hash_id, content_updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
"""
content_df = conn.execute(content_query).df()

signal_df = signal_df.merge(content_df, on="content_hash_id", how="left")
signal_df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(signal_df["content_updated_date"])).dt.days
signal_df["is_declining"] = (signal_df["imp_last"] < 0.8 * signal_df["imp_prev"]).astype(int)

print(f"Rows: {len(signal_df)} | Decline rate (base rate): {signal_df['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 76837 | Decline rate (base rate): 0.182


In [21]:
signal_df["staleness_bucket"] = pd.cut(
    signal_df["days_since_update"],
    bins=[-1, 30, 90, 180, 10000],
    labels=["0-30d", "31-90d", "91-180d", "180d+"]
)
staleness_check = signal_df.groupby("staleness_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
)
print(staleness_check)
print(f"\nBase rate: {signal_df['is_declining'].mean():.3f}")
# Verdict: fill in CONFIRMED / OPPOSITE / MIXED / FALSE based on whether decline_rate
# rises meaningfully across buckets, given n per bucket clears ~50

                      n  decline_rate
staleness_bucket                     
0-30d                77      0.506494
31-90d            16637      0.228407
91-180d              58      0.034483
180d+                27      0.666667

Base rate: 0.182


/tmp/ipykernel_657/379070412.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_check = signal_df.groupby("staleness_bucket").agg(


In [22]:
print("Rows with nonzero action_score:", (signal_df["action_score"] > 0).sum())

KeyError: 'action_score'

In [ ]:
print(signal_df["days_since_update"].describe())
print(signal_df["days_since_update"].quantile([0.5, 0.75, 0.9, 0.95]))

In [ ]:
valid_staleness = signal_df[signal_df["days_since_update"] >= 0]
print(f"Rows with a valid (pre-March-31) update date: {len(valid_staleness)} of {len(signal_df)}")
print(valid_staleness["days_since_update"].describe())
print(valid_staleness["days_since_update"].quantile([0.5, 0.75, 0.9]))

In [ ]:
# Staleness dropped from the primary score — confirmed unreliable above (97% bulk-artifact, majority future-dated)
was_visible = (signal_df["imp_prev"] >= 100).astype(int)  # already globally true, kept for clarity/reuse elsewhere
losing_ground = (signal_df["avg_position_mar"] > 10).astype(int)
has_valid_position = signal_df["avg_position_mar"].notna().astype(int)

signal_df["action_score"] = was_visible * has_valid_position * losing_ground * signal_df["imp_prev"]

def reason_code(row):
    if pd.isna(row["avg_position_mar"]):
        return "no_position_data"
    elif row["avg_position_mar"] > 10:
        return "visible_but_slipping"
    else:
        return "visible_and_ranked_well"

signal_df["reason_code"] = signal_df.apply(reason_code, axis=1)
signal_df["action"] = np.where(signal_df["action_score"] > 0, "review_for_refresh", "no_action")

print("Rows with nonzero action_score:", (signal_df["action_score"] > 0).sum(), "of", len(signal_df))

In [ ]:
print(valid_staleness["content_updated_date"].value_counts().head(10))

Verdict: FALSE. Staleness, as measured by dim_content.content_updated_date, does not hold up as a usable signal — for two compounding reasons. First, 60,038 of 76,837 rows (78%) have a content_updated_date after March 31, meaning "days since update" can't be honestly computed at the March decision point for most of the dataset — using it would risk leaking future information. Second, of the rows where it can be computed, 16,357 of 16,799 (97.4%) share the identical date 2026-02-25 — a bulk migration/re-publish event, not organic editorial activity — leaving only ~442 rows with any real, distinguishing staleness information. The bucket table's own numbers hinted at this before the root cause was found: the trend was non-monotonic (0-30d showed a higher decline rate, 0.506, than 31-90d's 0.228 — backwards from the plain-language rule), and two of four buckets (n=58, n=27) sat at or under the sample-size floor. A clearly negative result — it saved the rule from being built on a mostly-artificial signal.

In [ ]:
signal_df["position_bucket"] = pd.cut(
    signal_df["avg_position_mar"],
    bins=[0, 3, 10, 20, 50, 10000],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
position_check = signal_df.groupby("position_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
)
print(position_check)

Verdict: MIXED. Decline rate rises directionally from top_3 (0.118) through striking (0.211), consistent with the plain-language claim that worse position associates with higher decline risk, and every bucket clears the sample-size floor comfortably (n = 1,590–36,289). But the trend isn't strictly monotonic — page_3_5 (0.171) dips below striking (0.211) before deep (0.209) rises again — so I'm treating this as directionally real but not a clean, uniform relationship. Good enough to anchor the rule on, with the caveat named.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
was_visible = (signal_df["imp_prev"] >= 100).astype(int)
losing_ground = (signal_df["avg_position_mar"] > 10).astype(int)
has_valid_position = signal_df["avg_position_mar"].notna().astype(int)

signal_df["action_score"] = was_visible * has_valid_position * losing_ground * signal_df["imp_prev"]

def reason_code(row):
    if pd.isna(row["avg_position_mar"]):
        return "no_position_data"
    elif row["avg_position_mar"] > 10:
        return "visible_but_slipping"
    else:
        return "visible_and_ranked_well"

signal_df["reason_code"] = signal_df.apply(reason_code, axis=1)
signal_df["action"] = np.where(signal_df["action_score"] > 0, "review_for_refresh", "no_action")

queue = signal_df.sort_values("action_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
print("Rows with nonzero action_score:", (queue['action_score'] > 0).sum(), "of", len(queue))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in (10, 20, 50):
    p = precision_at_k(queue["action_score"].values, queue["is_declining"].values, k)
    print(f"Precision@{k}: {p:.3f}  (base rate: {signal_df['is_declining'].mean():.3f})")

Wrote 76837 rows to work/outputs/baseline_action_score.csv
Rows with nonzero action_score: 34526 of 76837
Precision@10: 0.100  (base rate: 0.182)
Precision@20: 0.200  (base rate: 0.182)
Precision@50: 0.160  (base rate: 0.182)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would
make it wrong.*

In [24]:
top10 = queue.head(10)[["content_hash_id", "days_since_update", "imp_prev", "avg_position_mar",
                          "action_score", "reason_code", "action", "is_declining"]]
print(top10.to_string(index=False))

         content_hash_id  days_since_update  imp_prev  avg_position_mar  action_score          reason_code             action  is_declining
content_9c057b66c30a3abb                 34  195648.0         11.967474      195648.0 visible_but_slipping review_for_refresh             1
content_e8a52cf3d5988c07                -72  162129.0         15.008339      162129.0 visible_but_slipping review_for_refresh             0
content_36e53e9c707674fc                -72  100736.0         32.766674      100736.0 visible_but_slipping review_for_refresh             0
content_df47d1b976106de4                -70   85935.0         24.355625       85935.0 visible_but_slipping review_for_refresh             0
content_84a6bf3578312e90                -78   79986.0         20.839925       79986.0 visible_but_slipping review_for_refresh             0
content_5e1c049f62e33b11                -72   73072.0         18.077081       73072.0 visible_but_slipping review_for_refresh             0
content_ba462518dad4

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [25]:
# Look for the weakest-looking picks in your top 20 by hand
print(queue.head(20)[["content_hash_id", "days_since_update", "imp_prev", "avg_position_mar", "reason_code"]])

# Leakage check: confirm nothing from March itself or beyond leaked into the RULE inputs
print("\nFeatures the rule used: days_since_update, imp_prev (Feb), avg_position_mar")
print("is_declining (label) computed from: imp_last (March) vs imp_prev (Feb) — imp_last never used as a feature.")
print("avg_position_mar is from the SAME month as the label window — flag this as a caveat, not a clean feature.")

                content_hash_id  days_since_update  imp_prev  \
24403  content_9c057b66c30a3abb                 34  195648.0   
65290  content_e8a52cf3d5988c07                -72  162129.0   
64864  content_36e53e9c707674fc                -72  100736.0   
64988  content_df47d1b976106de4                -70   85935.0   
11772  content_84a6bf3578312e90                -78   79986.0   
65393  content_5e1c049f62e33b11                -72   73072.0   
12945  content_ba462518dad435fc                -93   69134.0   
64424  content_3df3f32f3fd58dea                -70   68216.0   
63144  content_b51957d7f4abe47e                -72   57558.0   
64357  content_bdf60c86117079be                -93   51346.0   
24298  content_0709f29e7f096e6d                 34   51000.0   
7849   content_0aaa197051f58d6f                -87   49903.0   
64872  content_a3a1317f7c2bc3dd                -94   47036.0   
42131  content_87b9c790d43001dc                -76   46002.0   
44252  content_a3b985d3ee8a219b         

## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.